In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/long-running-agents-mistral/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# Long-running agents — the core idea, worked (Mistral edition)

**A long-running agent is not a long-running process.** It wakes up, does *one* step, checkpoints, and goes back to sleep. The store is the only memory; a queue delivers wake-ups; waiting means *parking* the run until something calls `resume()`.

Five rules keep it safe — find the `## (n)` markers in `durable.py`:
1. **durable state** — every step ends in `save()`
2. **intent → act** — journal the call *with an idempotency key*, save, *then* do it; a retry repeats the same call and never re-asks the model
3. **lease** — one worker per run; leases *expire*
4. **budget** — a hard step limit in code
5. **park, don't wait** — a wait is a status + token, not a sleeping process

Sections 1–7 run the stdlib core. Sections 8–9 show the two Mistral pieces: the model adapter and the same agent as a **Mistral Workflow** (Temporal underneath).

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))                      # repo root (run from notebooks/)
from durable import Agent, Crash, FakeClock, FakeModel, LeaseHeld, PaymentAPI, Queue, Store, Wait
RUNS = "runs.json"

def fresh(script, tools=None, clock=None):
    if os.path.exists(RUNS): os.remove(RUNS)
    pay = PaymentAPI()
    return Agent(Store(RUNS), Queue(), FakeModel(script), tools or {"charge": pay}, clock=clock or FakeClock()), pay

def show(run):
    print(f"{run.id}  status={run.status}  waiting_on={run.waiting_on}  result={run.result}")
    for i, s in enumerate(run.journal):
        print(f"  [{i}] {s['type']:<8}", {k: v for k, v in s.items() if k != "type"})

## 1. Happy path — one step per wake-up

In [2]:
agent, pay = fresh([{"tool": "charge", "args": {"amount": 42}}, {"final": "charged 42"}])
run = agent.start("pay invoice 42")
while agent.queue.deliver_one(agent.handle):                 # each delivery = one HTTP request / one task in prod
    print(f"wake-up #{agent.queue.delivered}: journal has {len(agent.store.get(run.id).journal)} entries, {len(agent.queue.items)} wake-up queued")
show(agent.store.get(run.id))
print("charges:", pay.charges)
print("on disk:", json.load(open(RUNS))[run.id]["status"])   # ← the only memory

wake-up #1: journal has 2 entries, 1 wake-up queued
wake-up #2: journal has 3 entries, 0 wake-up queued
run_1c3484  status=DONE  waiting_on=None  result=charged 42
  [0] decision {'tool': 'charge', 'args': {'amount': 42}}
  [1] intent   {'tool': 'charge', 'args': {'amount': 42}, 'key': 'run_1c3484:1', 'done': True, 'approved': False, 'result': {'charge_id': 'run_1c3484:1', 'amount': 42}}
  [2] decision {'final': 'charged 42'}
charges: {'run_1c3484:1': 42}
on disk: DONE


Read the journal: **decision → intent (with key `run:index`) → decision**. The intent line is what makes retries safe.

## 2. Crash *after* the card is charged, *before* the checkpoint

In [3]:
clock = FakeClock()
agent, pay = fresh([{"tool": "charge", "args": {"amount": 42}}, {"final": "charged 42"}], clock=clock)
run = agent.start("pay invoice 42")
agent.crash_at.add("after_side_effect")
try:
    agent.queue.deliver_one(agent.handle)
except Crash as e:
    print("💥", e)
show(agent.store.get(run.id))
print("charges:", pay.charges, "← money moved; journal says: intent, not done")

💥 process died after_side_effect
run_9ddd42  status=RUNNING  waiting_on=None  result=None
  [0] decision {'tool': 'charge', 'args': {'amount': 42}}
  [1] intent   {'tool': 'charge', 'args': {'amount': 42}, 'key': 'run_9ddd42:1', 'done': False, 'approved': False}
charges: {'run_9ddd42:1': 42} ← money moved; journal says: intent, not done


In [4]:
worker2 = Agent(agent.store, agent.queue, agent.model, agent.tools, worker="worker-2", clock=clock)
try:
    agent.queue.deliver_one(worker2.handle)                  # the queue retries — too early
except LeaseHeld as e:
    print("retry refused:", e)
clock.advance(61)                                            # the dead worker's lease expires
agent.queue.drain(worker2.handle)                            # re-executes the SAME intent with the SAME key
show(agent.store.get(run.id))
print("charges:", pay.charges, "| model calls:", agent.model.calls, "← one charge, model never re-asked")

retry refused: run_9ddd42 is held by worker-1 until 1060
run_9ddd42  status=DONE  waiting_on=None  result=charged 42
  [0] decision {'tool': 'charge', 'args': {'amount': 42}}
  [1] intent   {'tool': 'charge', 'args': {'amount': 42}, 'key': 'run_9ddd42:1', 'done': True, 'approved': False, 'result': {'charge_id': 'run_9ddd42:1', 'amount': 42}}
  [2] decision {'final': 'charged 42'}
charges: {'run_9ddd42:1': 42} | model calls: 2 ← one charge, model never re-asked


Why not just re-ask the model on retry? Because it might answer differently (`amount=42.5`), producing a *second, different* charge. Journaling the decision turns a probabilistic step into a replayable one.

## 3. Duplicate delivery (queues are at-least-once)

In [5]:
agent, pay = fresh([{"tool": "charge", "args": {"amount": 42}}, {"final": "charged 42"}])
run = agent.start("pay invoice 42")
agent.queue.duplicate_next()                                 # same message, delivered twice
agent.queue.drain(agent.handle)
print(agent.store.get(run.id).status, "| journal:", len(agent.store.get(run.id).journal), "| charges:", pay.charges, "| model calls:", agent.model.calls)

DONE | journal: 3 | charges: {'run_10af31:1': 42} | model calls: 2


## 4. Budget — the real stop condition

In [6]:
agent, pay = fresh([{"tool": "charge", "args": {"amount": 1}}] * 100)
run = agent.start("loop forever", max_steps=3)
agent.queue.drain(agent.handle)
print(agent.store.get(run.id).status, agent.store.get(run.id).result, "| charges:", len(pay.charges))

FAILED budget: 3 steps | charges: 3


## 5. Human gate — park with a token, execute exactly what was approved

In [7]:
pay = PaymentAPI(); pay.needs_approval = True
agent, _ = fresh([{"tool": "charge", "args": {"amount": 4200}}, {"final": "paid"}], tools={"charge": pay})
run = agent.start("pay invoice 4200")
agent.queue.drain(agent.handle)
run = agent.store.get(run.id)
print("parked:", run.status, run.waiting_on, "| queue:", agent.queue.items, "← nothing runs, nothing costs")
agent.resume(run.id, "wrong-token", {"approved": True}); print("wrong token →", agent.store.get(run.id).status)
agent.resume(run.id, run.waiting_on["token"], {"approved": True})
agent.resume(run.id, run.waiting_on["token"], {"approved": True})   # double click → no-op
agent.queue.drain(agent.handle)
show(agent.store.get(run.id)); print("charges:", pay.charges, "| model calls:", agent.model.calls)

parked: WAITING {'token': 'run_3e4a78:1', 'why': 'approval'} | queue: [] ← nothing runs, nothing costs
wrong token → WAITING
run_3e4a78  status=DONE  waiting_on=None  result=paid
  [0] decision {'tool': 'charge', 'args': {'amount': 4200}}
  [1] intent   {'tool': 'charge', 'args': {'amount': 4200}, 'key': 'run_3e4a78:1', 'done': True, 'approved': True, 'result': {'charge_id': 'run_3e4a78:1', 'amount': 4200}}
  [2] decision {'final': 'paid'}
charges: {'run_3e4a78:1': 4200} | model calls: 2


## 6. Slow tool — the job runs elsewhere; the webhook resumes the run

In [8]:
def export(region, key):
    return Wait(token="job-" + key)                          # returns a handle, not a result
agent, _ = fresh([{"tool": "export", "args": {"region": "apac"}}, {"final": "report ready"}], tools={"export": export})
run = agent.start("export apac")
agent.queue.drain(agent.handle)
run = agent.store.get(run.id); print("parked:", run.status, run.waiting_on)
agent.resume(run.id, run.waiting_on["token"], {"rows": 1200})       # what the webhook (or a poller) does
agent.queue.drain(agent.handle)
show(agent.store.get(run.id))

parked: WAITING {'token': 'job-run_034986:1', 'why': 'event'}
run_034986  status=DONE  waiting_on=None  result=report ready
  [0] decision {'tool': 'export', 'args': {'region': 'apac'}}
  [1] intent   {'tool': 'export', 'args': {'region': 'apac'}, 'key': 'run_034986:1', 'done': True, 'approved': False, 'result': {'rows': 1200}}
  [2] decision {'final': 'report ready'}


## 7. A real crash, across two processes
From a terminal in the repo root: `python demo.py kill` (charges, then `os._exit(137)`), then `python demo.py resume` — a fresh process finds the run in `runs.json` and finishes it with one charge.

## 8. Swap the model: a Mistral model decides
`mistral_model.MistralModel` rebuilds the journal as a chat — every earlier decision as an assistant **tool call**, every recorded result as the matching **tool result** — and asks the model via function calling. Same dict shape out, so `durable.py` does not change. Offline here with a fake client; `python ../demo.py live` uses the real API.

In [9]:
from types import SimpleNamespace as NS
from mistral_model import TOOL_SPECS, MistralModel, to_messages, function_tools

journal = [{"type": "decision", "tool": "charge", "args": {"amount": 42}},
           {"type": "intent", "tool": "charge", "args": {"amount": 42}, "key": "run_x:1", "done": True, "approved": True,
            "result": {"charge_id": "run_x:1", "amount": 42}}]
for m in to_messages("pay invoice 42", journal):
    print(m["role"], "→", {k: v for k, v in m.items() if k != "role"})
print("tools sent to the model:", json.dumps(function_tools(TOOL_SPECS))[:120], "…")

class FakeMistral:                                            # speaks the SDK's response shape
    def __init__(self, script): self.script, self.chat = list(script), self
    def complete(self, **kw):
        item = self.script.pop(0)
        msg = (NS(content="", tool_calls=[NS(id="abc123def", function=NS(name=item["tool"], arguments=json.dumps(item["args"])))])
               if "tool" in item else NS(content=item["final"], tool_calls=None))
        return NS(choices=[NS(message=msg)])

model = MistralModel(TOOL_SPECS, model="mistral-medium-latest", client=FakeMistral([{"tool": "charge", "args": {"amount": 42}}, {"final": "Charged 42."}]))
pay = PaymentAPI()
if os.path.exists(RUNS): os.remove(RUNS)
agent = Agent(Store(RUNS), Queue(), model, {"charge": pay})
run = agent.start("pay invoice 42"); agent.queue.drain(agent.handle)
show(agent.store.get(run.id)); print("charges:", pay.charges)

system → {'content': 'You are an operations agent working one step at a time. Use a tool when the goal needs one; when the goal is met, reply with a short plain-text summary and no tool call. Never repeat a tool call whose result you already have.'}
user → {'content': 'pay invoice 42'}
assistant → {'content': '', 'tool_calls': [{'id': '1380691d0', 'type': 'function', 'function': {'name': 'charge', 'arguments': '{"amount": 42}'}}]}
tool → {'tool_call_id': '1380691d0', 'name': 'charge', 'content': '{"charge_id": "run_x:1", "amount": 42}'}
tools sent to the model: [{"type": "function", "function": {"name": "charge", "description": "Charge the customer's card. Requires human approval …
run_870b17  status=DONE  waiting_on=None  result=Charged 42.
  [0] decision {'tool': 'charge', 'args': {'amount': 42}}
  [1] intent   {'tool': 'charge', 'args': {'amount': 42}, 'key': 'run_870b17:1', 'done': True, 'approved': False, 'result': {'charge_id': 'run_870b17:1', 'amount': 42}}
  [2] decision {'fina

Two API details baked into the adapter: Mistral tool-call ids must be **9 alphanumeric characters** (the journal key is hashed to one), and the SDK is now a namespace package (`from mistralai.client import Mistral`).

## 9. The same agent as a Mistral Workflow
`mistral_workflow.py` is `durable.py` with the plumbing removed, because **Mistral Workflows (Temporal) provides the five rules**:

| Rule | In `durable.py` | On Mistral Workflows |
|---|---|---|
| 1 durable state | `Store.save()` after each step | the execution's event history; a replacement worker replays it |
| 2 intent → act | journal `intent` + key, save, then act | each **activity** call is recorded, retried on failure, never re-run once complete — the model call is an activity too |
| 3 lease | `acquire_lease` with TTL | one worker per task; missed heartbeats → rescheduled |
| 4 budget | `max_steps` in `_decide` | a loop bound in deterministic workflow code + `execution_timeout` |
| 5 park | `WAITING` + token, `resume()` | `workflow.wait_condition(...)` + `@workflow.signal` — zero cost while parked |

Run it: `python ../demo.py workflow` (a Temporal dev server is downloaded on first use), or `python -m pytest ../tests/test_workflow.py`. Read the source:

In [10]:
import inspect, mistral_workflow
src = inspect.getsource(mistral_workflow)
print(src[src.index("# --- activities"):])

2026-09-19T12:31:46.358314Z [info     ] Configuration loaded           [mistralai.workflows.core.config.config] config={'common': {'app_name': 'mistral-workflows', 'app_version': '0.0.0', 'log_format': <LogFormat.CONSOLE: 'console'>, 'log_level': <LogLevel.INFO: 'INFO'>, 'otel_enabled': True, 'mistral_workflows_otel_traces_export': True, 'mistral_workflows_otel_metrics_export': True, 'mistral_workflows_otel_logs_export': True, 'otel_endpoint': None, 'otel_traces_endpoint': None, 'otel_metrics_endpoint': None, 'otel_logs_endpoint': None, 'otel_sample_rate': 1.0, 'otel_export_interval_ms': 30000, 'temporal_runtime_metrics_buffer_size': 30000, 'temporal_runtime_metrics_drain_interval_s': 5.0, 'otel_local': False, 'otel_inject_logs': True, 'otel_redaction': <OtelRedactionMode.DEFAULT: 'default'>, 'ca_bundle': None, 'mistral_api_key': None, 'mistral_sa_token_path': None}, 'worker': {'retry_policy_max_attempts': 3, 'retry_policy_backoff_coefficient': 2.0, 'dangerously_force_fail_workflow_on_

2026-09-19T12:31:46.540065Z [info     ] Workflow registered WITHOUT versioning behavior [mistralai.workflows.core.workflow] lineno=289 pathname=/usr/local/lib/python3.12/dist-packages/mistralai/workflows/core/workflow.py workflow_name=__parallel_execution__


2026-09-19T12:31:46.544089Z [info     ] Workflow registered WITHOUT versioning behavior [mistralai.workflows.core.workflow] lineno=289 pathname=/usr/local/lib/python3.12/dist-packages/mistralai/workflows/core/workflow.py workflow_name=__parallel_execution_obo__


2026-09-19T12:31:46.600371Z [info     ] Workflow registered WITHOUT versioning behavior [mistralai.workflows.core.workflow] lineno=289 pathname=/usr/local/lib/python3.12/dist-packages/mistralai/workflows/core/workflow.py workflow_name=invoice-agent


# --- activities: the only place side effects happen -------------------------
@workflows.activity(start_to_close_timeout=timedelta(minutes=2), retry_policy_max_attempts=5)
async def decide(goal: str, journal: list) -> dict:
    """One model call. Recorded in history → a replay reuses the recorded decision. (2)"""
    return MODEL.decide(goal, journal)


@workflows.activity(start_to_close_timeout=timedelta(seconds=30), retry_policy_max_attempts=5,
                    retry_policy_backoff_coefficient=2.0)
async def charge(amount: float, key: str) -> dict:
    """Idempotent by key. If the worker dies after the API call, the retry repeats the SAME
    call with the SAME key and the payment API de-duplicates. (2)"""
    result = PAYMENTS(amount, key)
    if CRASH_ONCE["after_charge"]:            # simulate: money moved, then the process died
        CRASH_ONCE["after_charge"] = False
        raise RuntimeError("worker died after the side effect")
    return result


TOOLS = {"charge": char

What changed from `durable.py`: no `Store`, no `Queue`, no lease, no `park`/`resume` bookkeeping — the orchestrator owns all of it. What did **not** change: the charge still carries an idempotency key (retries repeat the activity), the model call is still recorded before anything acts on it, and the budget is still code.

Hosted mode: `MISTRAL_API_KEY` + `workflows.run_worker([InvoiceAgent])` in your own infrastructure; Mistral hosts the orchestrator; start executions and send the `decide` signal from the Studio console or the Workflows API. Enterprise deployments can self-host the orchestrator too.